# YOGA quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TrustAIoT/YOGA_object-detection/blob/main/YOGA_quickstart.ipynb)

**YOGA: Deep Object Detection in the Wild with Lightweight Feature Learning and Multiscale Attention**
Raja Sunkara and Tie Luo, *Pattern Recognition* 2023 · [paper](https://doi.org/10.1016/j.patcog.2023.109451) · [arXiv](https://arxiv.org/abs/2307.05945) · [code](https://github.com/TrustAIoT/YOGA_object-detection)

This notebook takes a few minutes and covers:
1. Install
2. Download the pretrained COCO weights
3. Detect objects in images
4. Use YOGA from Python through PyTorch Hub
5. Measure accuracy (mAP) on COCO128
6. Fine-tune on your own data

For a GPU, use *Runtime → Change runtime type → T4 GPU*. Everything also runs on CPU, just more slowly.

## 1. Install

In [ ]:
!git clone -q https://github.com/TrustAIoT/YOGA_object-detection
%cd YOGA_object-detection
!pip install -q -r requirements.txt gdown

import torch
print(f"PyTorch {torch.__version__}, GPU: {torch.cuda.is_available()}")

## 2. Download pretrained weights

There are four sizes:

| Model | AP (COCO val) | Params | FLOPs |
|---|:-:|:-:|:-:|
| YOGA-n | 32.3 | 1.9M | 4.9B |
| YOGA-s | 40.7 | 7.6M | 16.6B |
| YOGA-m | 45.2 | 16.3M | 34.6B |
| YOGA-l | 48.9 | 33.6M | 71.8B |

The cell tries the GitHub release first. If that fails, it downloads the Google Drive folder.

In [ ]:
SIZE = "s"  # n, s, m or l

import glob, os, re, shutil, subprocess
weights = f"YOGA-{SIZE}.pt"
url = f"https://github.com/TrustAIoT/YOGA_object-detection/releases/download/v1.0/{weights}"
if not os.path.exists(weights):
    subprocess.run(["wget", "-q", url, "-O", weights])
    if os.path.getsize(weights) < 100_000:  # release not available: use Google Drive
        os.remove(weights)
        subprocess.run(["gdown", "-q", "--folder", "1K2rKYY9p3wmA6-rN0Jx8g_6pBazyGZjO", "-O", "gdrive_weights"])
        found = [p for p in glob.glob("gdrive_weights/**/*.pt", recursive=True)
                 if re.search(rf"(^|[^a-z]){SIZE}([^a-z]|$)", os.path.basename(p).lower().replace(".pt", ""))]
        print("Found in Google Drive:", glob.glob("gdrive_weights/**/*.pt", recursive=True))
        shutil.copy(found[0], weights)
print("Using", weights, f"({os.path.getsize(weights) / 1e6:.1f} MB)")

## 3. Detect objects in images

In [ ]:
!mkdir -p samples
!wget -q https://ultralytics.com/images/bus.jpg -O samples/bus.jpg
!wget -q https://ultralytics.com/images/zidane.jpg -O samples/zidane.jpg
!python detect.py --weights {weights} --source samples --conf-thres 0.25 --project runs/detect --name demo --exist-ok

from IPython.display import Image, display
for f in ["bus.jpg", "zidane.jpg"]:
    display(Image(f"runs/detect/demo/{f}", width=600))

Other sources work the same way: `--source video.mp4`, `--source path/to/folder`, or `--source 0` for a webcam.

## 4. PyTorch Hub

Load YOGA from any Python script, with no need to clone the repo:

In [ ]:
model = torch.hub.load("TrustAIoT/YOGA_object-detection", "custom", path=weights, trust_repo=True)
results = model("samples/zidane.jpg")
results.print()
results.pandas().xyxy[0]  # boxes as a DataFrame

## 5. Measure accuracy on COCO128

COCO128 is the first 128 images of COCO train2017 (about 7 MB), so it's a quick check. For the paper's numbers, evaluate on the full COCO val2017 with `bash data/scripts/get_coco.sh` (about 20 GB).

In [ ]:
!python val.py --weights {weights} --data coco128.yaml --img 640 --iou 0.65 --project runs/val --name coco128 --exist-ok

## 6. Fine-tune on your own data

Write a dataset YAML in the YOLOv5 format (image folders, labels in `.txt` files and class names; see `data/coco128.yaml`), then start from the COCO weights. This short run on COCO128 shows the command:

In [ ]:
HYP = {"n": "n", "s": "s", "m": "m", "l": "m"}[SIZE]  # YOGA-l was trained with the YOGA-m hyperparameters
!python train.py --data coco128.yaml --cfg models/YOGA-{SIZE}.yaml --hyp data/hyps/YOGA-{HYP}.yaml --weights {weights} \
    --img 640 --batch-size 16 --epochs 3 --project runs/train --name finetune --exist-ok
display(Image("runs/train/finetune/results.png", width=800))

## Citation

If YOGA or its building blocks (CSPGhost, AFF-PANet) help your work, please cite:

```bibtex
@article{sunkara2023yoga,
  title   = {{YOGA}: Deep Object Detection in the Wild with Lightweight Feature Learning and Multiscale Attention},
  author  = {Sunkara, Raja and Luo, Tie},
  journal = {Pattern Recognition},
  volume  = {139},
  pages   = {109451},
  year    = {2023},
  doi     = {10.1016/j.patcog.2023.109451}
}
```